In [101]:
import pandas as pd

csv_file_path = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\subsector_workflows\industry\data_preparation\input_data\industrial_database.csv"
df = pd.read_csv(csv_file_path)
df.drop(columns=['y', 'x', "ID"], inplace=True, errors='ignore')

countries =["MA","EG","ZA","CL"]
df_filtered = df[df['country'].isin(countries)]
df_filtered = df_filtered.groupby(['country', 'technology', 'unit']).sum(numeric_only=True).reset_index()
df_filtered

,country,technology,unit,capacity
0,CL,Cement,Kt/yr,1.099800e+04
1,CL,Electric arc,kt/yr,5.200000e+02
2,CL,HVC,bpd,2.267500e+05
3,CL,Integrated steelworks,kt/yr,1.450000e+03
4,CL,Paper,10kt/yr,1.416000e+06
5,EG,Cement,Kt/yr,7.383411e+04
6,EG,DRI + Electric arc,kt/yr,1.060000e+04
7,EG,Electric arc,kt/yr,3.400000e+03
8,EG,HVC,bpd,6.770000e+05
9,MA,Cement,Kt/yr,2.701385e+04


In [ ]:
csv_file_path = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\subsector_workflows\industry\data_preparation\input_data\industrial_database.csv"
df = pd.read_csv(csv_file_path)

# --- Define feedstock shares and conversion factors (in kt/year per 1,000 bpd) ---
feedstock_shares = {
    "naphtha": 0.60,
    "LPG": 0.25,
    "methanol": 0.15
} # based on @EDP assumptions

conversion_factors_kt_per_kbd = {
    "naphtha": 41.6,
    "LPG": 33.2,
    "methanol": 46.0
} # TODO: validate the conversion factors per fuel

# --- Calculate weighted average conversion factor in t/year per 1,000 bpd ---
weighted_t_per_kbd = sum(
    feedstock_shares[fuel] * conversion_factors_kt_per_kbd[fuel] for fuel in feedstock_shares
) * 1000  # convert kt/year to t/year

# --- Filter HVC technologies ---
hvc_df = df[df["technology"] == "HVC"].copy()

# --- Calculate t/year assuming capacity is in 1,000 bpd ---
hvc_df["capacity_t_per_year"] = hvc_df["capacity"] * weighted_t_per_kbd

# --- Filter only countries of interest ---
target_countries = ["MA", "EG", "CL", "ZA"]
filtered = hvc_df[hvc_df["country"].isin(target_countries)]

# --- Summarize results per country ---
result = filtered.groupby("country")["capacity_t_per_year"].sum().reset_index()
result.columns = ["country", "HVC_production_t_per_year"]

# --- Display result ---
print("Feedstock Shares:", feedstock_shares)
print("Conversion Factors (kt/year per 1,000 bpd):", conversion_factors_kt_per_kbd)
print("\nEstimated HVC Production (t/year):")
result



Feedstock Shares: {'naphtha': 0.6, 'LPG': 0.25, 'methanol': 0.15}
Conversion Factors (kt/year per 1,000 bpd): {'naphtha': 41.6, 'LPG': 33.2, 'methanol': 46.0}

Estimated HVC Production (t/year):


,country,HVC_production_t_per_year
0,CL,9.106280e+09
1,EG,2.718832e+10
2,MA,6.220784e+09
3,ZA,1.993944e+10


In [112]:
cement_df = df_filtered[df_filtered['technology'].str.lower() == 'cement'].copy()
# Convert all Kt/yr and kt/yr to t/yr
mask_kt = cement_df['unit'].isin(['Kt/yr', 'kt/yr'])
cement_df.loc[mask_kt, 't/year'] = cement_df.loc[mask_kt, 'capacity']*0.7 * 1000
# If there are other units, handle them as needed
cement_df

,country,technology,unit,capacity,t/year
0,CL,Cement,Kt/yr,10998.000000,7.698600e+06
5,EG,Cement,Kt/yr,73834.105263,5.168387e+07
9,MA,Cement,Kt/yr,27013.846154,1.890969e+07
12,ZA,Cement,Kt/yr,35866.666667,2.510667e+07


In [114]:
paper = df_filtered[df_filtered['technology'].str.lower() == 'paper'].copy()
paper

,country,technology,unit,capacity
4,CL,Paper,10kt/yr,1416000.0
